# `parcelsim` — NYC & Lyon demand maps

Full pipeline walkthrough. Zones are built from a 1 km² grid — no downloads needed.

In [ ]:
!pip install -q "parcelsim[viz]"

In [ ]:
%matplotlib inline
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import contextily as ctx
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from shapely.geometry import Point, box
import parcelsim
print(f"parcelsim {parcelsim.__version__}")

In [ ]:
from parcelsim.city import City
from parcelsim.population.base import SyntheticPopulation
from parcelsim.operators.operator import OperatorRegistry
from parcelsim.operators.assignment import assign_parcels

def build_grid_city(name, country_iso, crs, cx_m, cy_m,
                    width_km, height_km, cell_km=1.0, base_hh=800, rng_seed=42):
    rng   = np.random.default_rng(rng_seed)
    cell  = cell_km * 1000
    half_w, half_h = (width_km * 1000) / 2, (height_km * 1000) / 2
    xs    = np.arange(cx_m - half_w, cx_m + half_w, cell)
    ys    = np.arange(cy_m - half_h, cy_m + half_h, cell)
    sigma = max(half_w, half_h) * 0.42
    rows  = []
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            cx_, cy_ = x + cell / 2, y + cell / 2
            dist    = np.sqrt((cx_ - cx_m) ** 2 + (cy_ - cy_m) ** 2)
            density = np.exp(-0.5 * (dist / sigma) ** 2)
            n_hh    = max(30, int(base_hh * density + rng.integers(-50, 50)))
            rows.append({
                "zone_id":      f"{name}_{i:03d}_{j:03d}",
                "population":   int(n_hh * 2.3),
                "n_households": n_hh,
                "area_km2":     cell_km ** 2,
                "centroid_x":   cx_, "centroid_y": cy_,
                "geometry":     box(x, y, x + cell, y + cell),
            })
    zones = gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)
    return City.from_zones(name=name, zones=zones, crs=crs, country_iso=country_iso)


OP_COLORS = {
    "usps":        "#27ae60",
    "ups":         "#c0392b",
    "fedex":       "#8e44ad",
    "amazon":      "#2980b9",
    "colissimo":   "#27ae60",
    "chronopost":  "#c0392b",
    "dpd":         "#e67e22",
    "dhl":         "#f39c12",
    "gls":         "#2980b9",
    "colis_prive": "#8e44ad",
}


def demand_map(demand, assignment, registry, title, zoom=11, figsize=(12, 10),
               n_grid=300, smooth_sigma=2.5, n_levels=12):
    crs = demand.zone_demand.crs
    zd  = demand.zone_demand.copy()

    xs = zd["centroid_x"].values
    ys = zd["centroid_y"].values
    zs = zd["n_delivery"].values

    xi = np.linspace(xs.min(), xs.max(), n_grid)
    yi = np.linspace(ys.min(), ys.max(), n_grid)
    Xi, Yi = np.meshgrid(xi, yi)

    Zi = griddata((xs, ys), zs, (Xi, Yi), method="cubic", fill_value=0.0)
    Zi = np.clip(Zi, 0, None)
    Zi = gaussian_filter(Zi, sigma=smooth_sigma)

    fig, ax = plt.subplots(figsize=figsize)

    # Set extent first so contextily fetches the right tiles
    pad_x = (xs.max() - xs.min()) * 0.04
    pad_y = (ys.max() - ys.min()) * 0.04
    ax.set_xlim(xs.min() - pad_x, xs.max() + pad_x)
    ax.set_ylim(ys.min() - pad_y, ys.max() + pad_y)

    # Basemap (bottom layer)
    ctx.add_basemap(ax, crs=crs,
                    source=ctx.providers.CartoDB.Positron, zoom=zoom,
                    zorder=1)

    # Isolines on top — alpha=0.50 keeps basemap visible
    cmap = plt.cm.YlOrRd
    cf = ax.contourf(Xi, Yi, Zi, levels=n_levels, cmap=cmap, alpha=0.50, zorder=2)
    ax.contour(Xi, Yi, Zi, levels=n_levels,
               colors="white", linewidths=0.4, alpha=0.5, zorder=3)

    cbar = fig.colorbar(cf, ax=ax, shrink=0.5, pad=0.02)
    cbar.set_label("Parcels / day", fontsize=10)

    # Depot markers
    depots = registry.all_depots(crs="EPSG:4326").to_crs(crs)
    for op in registry.operators:
        op_dep = depots[depots["operator_id"] == op.operator_id]
        if op_dep.empty:
            continue
        ax.scatter(
            op_dep.geometry.x, op_dep.geometry.y,
            s=120, color=OP_COLORS.get(op.operator_id, "#333"),
            edgecolors="white", linewidths=0.8, zorder=4,
            label=op.operator_id.upper()
        )

    ax.set_title(title, fontsize=13, pad=10)
    ax.set_axis_off()
    ax.legend(title="Depots", loc="lower right",
              fontsize=8, title_fontsize=9, framealpha=0.85)
    plt.tight_layout()
    plt.show()

print("ready")

---
## NYC — US demand model

In [ ]:
from parcelsim.demand.generic_model import AggregateDemandModel

CRS_NYC = "EPSG:32618"
nyc_city = build_grid_city("nyc", "US", CRS_NYC,
                            cx_m=585_000, cy_m=4_511_000,
                            width_km=28, height_km=24, base_hh=1400)

hh_rows = []
for _, zone in nyc_city.zones.iterrows():
    hh_rows.append({
        "household_id": f"{zone['zone_id']}_0",
        "zone_id":       zone["zone_id"],
        "geometry":      Point(zone["centroid_x"], zone["centroid_y"]),
        "n_persons": 2, "income_bracket": "median",
        "n_households":  int(zone["n_households"]),
    })

nyc_hh  = gpd.GeoDataFrame(hh_rows, geometry="geometry", crs=CRS_NYC)
nyc_pop = SyntheticPopulation(city=nyc_city, households=nyc_hh,
                               source_adapter="synthetic", year=2021)

nyc_demand     = AggregateDemandModel.from_country("US", demand_factor=1.114).generate(nyc_pop)
nyc_registry   = OperatorRegistry.from_builtin("us_2021")
nyc_assignment = assign_parcels(nyc_demand, nyc_registry, nyc_city)

print(f"{len(nyc_city.zones)} zones")
print(nyc_demand.summary())
print(nyc_assignment.summary())

In [ ]:
demand_map(nyc_demand, nyc_assignment, nyc_registry,
           title="Average daily delivery volume by zone — NYC (synthetic grid)")

---
## Lyon — France demand model (Hörl et al. 2025)

In [ ]:
from parcelsim.demand.france_model import FranceDemandModel

CRS_LYON = "EPSG:2154"
lyon_city = build_grid_city("lyon", "FR", CRS_LYON,
                             cx_m=843_000, cy_m=6_519_000,
                             width_km=30, height_km=22, base_hh=900)

lyon_hh_rows = []
for _, zone in lyon_city.zones.iterrows():
    lyon_hh_rows.append({
        "household_id": f"{zone['zone_id']}_0",
        "zone_id":       zone["zone_id"],
        "geometry":      Point(zone["centroid_x"], zone["centroid_y"]),
        "n_persons": 2, "income_bracket": "median",
        "n_households":  int(zone["n_households"]),
    })

lyon_hh  = gpd.GeoDataFrame(lyon_hh_rows, geometry="geometry", crs=CRS_LYON)
lyon_pop = SyntheticPopulation(city=lyon_city, households=lyon_hh,
                                source_adapter="worldpop", year=2020)

lyon_demand     = FranceDemandModel(demand_factor=1.35).generate(lyon_pop)
lyon_registry   = OperatorRegistry.from_builtin("lyon_2024")
lyon_assignment = assign_parcels(lyon_demand, lyon_registry, lyon_city)

print(f"{len(lyon_city.zones)} zones")
print(lyon_demand.summary())
print(lyon_assignment.summary())

In [ ]:
demand_map(lyon_demand, lyon_assignment, lyon_registry,
           title="Average daily delivery volume by zone — Lyon Métropole (synthetic grid)")